# Espacio de trabajo
Las rutas se resuelven desde el repositorio. Los datos y resultados quedan en el disco de la máquina de investigación.

In [ ]:
import hashlib
import json
import os
import re
from pathlib import Path
from dotenv import load_dotenv

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)
load_dotenv(ROOT / ".env")
DATA = ROOT / "data"
RUNS = ROOT / "artifacts" / "runs"


def digest(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def save_json(path, value):
    path.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8",
    )


def dataset_location():
    workspace, project, version = (
        os.getenv("ROBOFLOW_" + key, "") for key in ("WORKSPACE", "PROJECT", "VERSION")
    )
    if (
        not all(re.fullmatch(r"[A-Za-z0-9_-]+", slug) for slug in (workspace, project))
        or not version.isdigit()
        or int(version) < 1
    ):
        raise ValueError(
            "Revisa el espacio de trabajo, el proyecto y la versión en .env."
        )
    return DATA / "roboflow" / workspace / project / version


RESULTS = ROOT / "results"


def export_file(source, target):
    from shutil import copyfile
    from tempfile import NamedTemporaryFile

    target.parent.mkdir(parents=True, exist_ok=True)
    with NamedTemporaryFile(dir=target.parent, delete=False) as stream:
        temporary = Path(stream.name)
    try:
        copyfile(source, temporary)
        temporary.replace(target)
    finally:
        temporary.unlink(missing_ok=True)


def export_run(run, record):
    folder = RESULTS / "runs" / run.name
    folder.mkdir(parents=True, exist_ok=True)
    summary = {key: value for key, value in record.items() if key not in {"cause", "train_files"}}
    summary["run_id"] = run.name
    summary["source_record_sha256"] = digest(run / "record.json")
    weights = run / "weights.pt"
    if weights.is_file():
        summary["weights_sha256"] = digest(weights)
    save_json(run / "summary.json", summary)
    export_file(run / "summary.json", folder / "record.json")
    for name in ("threshold.json", "public-test/metrics.json", "valid/metrics.json"):
        if (run / name).is_file():
            export_file(run / name, folder / name)


def export_study_file(folder, name):
    relative = folder.relative_to(RUNS.parent / "final")
    export_file(folder / name, RESULTS / "studies" / relative / name)
